# Sparkle V2 — Fase 3.2b: Fusión apariencia + geometría (el experimento que decide el proyecto)

**Entorno de ejecución:** Google Colab (GPU T4) o Kaggle Notebooks (GPU)
**Requiere:** embeddings ya extraídos por `Sparkle_V2_Fase3_2a_Extraccion_Embeddings_CNN.ipynb` (`embeddings_cnn/{Train,Validation,Test}/*.npz`) + las features geométricas de `features_v2/` (mismas de Fase 3.0/3.1).

**Objetivo (Roadmap Fase 3.2, semanas 3-4 del plan):** entrenar el modelo de fusión (§3.3) y correr la ablación **embeddings-solo vs geométricas-solo vs fusión** — esta es la evidencia central del diagnóstico del plan (§0): la apariencia es la palanca principal, la geometría por sí sola tiene un techo ~50%.

**Criterio de salida:** ≥ 68–72% accuracy binaria en Test oficial con brecha > +12 pts vs. trivial. Si se cumple, es el resultado que sostiene la tesis (según el plan, "el experimento que decide el proyecto").

## Cómo correr esto en Colab

1. `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → **T4 GPU**.
2. Ejecutar la celda de setup: monta Drive y verifica que estén los datos.
3. `Entorno de ejecución` → `Ejecutar todas`.

**No hace falta `pip install`**: Colab ya trae TensorFlow, Keras 3, scikit-learn y pandas. Instalar/reinstalar TensorFlow rompe la sesión y obliga a reiniciar el runtime.

> **Adaptación a Keras 3.** Colab hoy corre TensorFlow 2.19 + Keras 3. Respecto de la versión original de este notebook (escrita contra Keras 2) cambian tres cosas, señalizadas en el código: (a) el export a TFLite pasa por `ExportArchive` → SavedModel → `from_saved_model` (en Keras 3 `TFLiteConverter.from_keras_model` ya no acepta modelos Keras directamente), (b) las capas custom declaran sus sub-capas en `__init__` y exponen `compute_output_shape`/`get_config`, (c) todas las capas llevan nombre explícito para que la copia de pesos al modelo de export sea verificable.

In [ ]:
# =====================================================================
# SETUP — detección de entorno, montaje de Drive y verificación de datos
# =====================================================================
import sys, os, glob, json, time

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
elif os.path.isdir('/kaggle/input'):
    PROJECT_DIR = '/kaggle/input/sparklev2'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2')

print('Entorno:', 'Colab' if IN_COLAB else ('Kaggle' if os.path.isdir('/kaggle/input') else 'local'))
print('PROJECT_DIR =', PROJECT_DIR)
assert os.path.isdir(PROJECT_DIR), (
    f'No existe {PROJECT_DIR}. En Colab la carpeta tiene que estar en "Mi unidad" '
    f'(los accesos directos a "Compartido conmigo" NO se montan). Ajustá PROJECT_DIR a mano si hace falta.')

FEATURES_DIR = f'{PROJECT_DIR}/features_v2'
EMB_DIR      = f'{PROJECT_DIR}/embeddings_cnn'
CKPT_DIR     = f'{PROJECT_DIR}/checkpoints_fase3_2'
os.makedirs(CKPT_DIR, exist_ok=True)

# Verificacion temprana: sin embeddings de 3.2a este notebook no puede correr
for split in ('Train', 'Validation', 'Test'):
    n_g = len(glob.glob(f'{FEATURES_DIR}/{split}/*.npz'))
    n_e = len(glob.glob(f'{EMB_DIR}/{split}/*.npz'))
    print(f'{split}: {n_g} geometricas | {n_e} embeddings')
    assert n_g > 0, f'Faltan geometricas en {FEATURES_DIR}/{split}'
    assert n_e > 0, (f'Faltan embeddings en {EMB_DIR}/{split}. '
                     f'Correr primero Sparkle_V2_Fase3_2a_Extraccion_Embeddings_CNN.ipynb')

import tempfile, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers
from collections import Counter
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

KERAS3 = keras.__version__.startswith('3')
gpus = tf.config.list_physical_devices('GPU')
print(f'\nTF {tf.__version__} | Keras {keras.__version__} | GPU: {gpus}')
if not gpus:
    print('\n*** SIN GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU ***\n')

SEED = 42
N_FRAMES = 60
N_EMB_FRAMES = 16
EMB_DIM = 128
np.random.seed(SEED); tf.random.set_seed(SEED)

MAX_EPOCHS = 120
PATIENCE   = 15
BATCH_SIZE = 128
SEEDS      = [0, 1, 2, 3, 4]
MODES      = ['geom_only', 'emb_only', 'fusion']

## 1. Carga y alineación de las dos fuentes (embeddings + geométricas)

Cada clip tiene un `.npz` de geométricas (`features_v2`, 22 features crudas) y otro de embeddings (`embeddings_cnn`, 16×128). Se cargan ambas por `ClipID` y se descartan los clips que falten en cualquiera de las dos fuentes (deben ser pocos: mismos videos, mismo detector facial subyacente).

**Por qué esta celda tiene caché y staging local.** Leer ~16.000 `.npz` chicos (8.000 geométricas + 8.000 embeddings) directamente desde `/content/drive/` tarda decenas de minutos: Drive se monta por FUSE y cada `np.load` paga latencia de red en la apertura del archivo. Dos mitigaciones, en orden:

1. **Caché consolidada.** Si ya existe `dataset_fase3_2_alineado.npz`, se carga eso: un archivo grande en vez de 16.000 chicos, cuestión de segundos.
2. **Staging local.** Si no existe, primero se copian las carpetas al disco del runtime (`/content`) y se lee de ahí. La copia también pasa por Drive, pero se paga una sola vez; después las 16.000 lecturas son locales.

Al terminar se guarda la caché, así las próximas sesiones arrancan por el camino 1.

> Nota: `mmap_mode='r'` **no** ayuda acá. `np.load` lo ignora cuando el archivo es un `.npz` (devuelve un `NpzFile` sin pasarle el parámetro), y además estos archivos se escribieron con `savez_compressed`: un archivo comprimido no se puede memory-mapear, hay que descomprimirlo entero. Tampoco es cierto que se lea "el archivo entero solo para consultar la shape" — `np.load` sobre un `.npz` es perezoso y la descompresión ocurre recién al acceder a la clave.

In [ ]:
from tqdm.auto import tqdm

DELTA_FEATURES = ['ear_avg', 'yaw', 'pitch', 'mar', 'jawOpen',
                  'eyeBlinkLeft', 'eyeBlinkRight',
                  'eyeLookInLeft', 'eyeLookOutLeft', 'browInnerUp']

CACHE_NPZ  = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
USAR_CACHE = True          # False = forzar relectura de los .npz individuales
os.makedirs(os.path.dirname(CACHE_NPZ), exist_ok=True)

def staging_local():
    '''Copia features_v2 y embeddings_cnn al disco del runtime. Devuelve las rutas
    locales (o las originales si no estamos en Colab).'''
    if not IN_COLAB:
        return FEATURES_DIR, EMB_DIR
    local = '/content/sparkle_local'
    destinos = {}
    for nombre, origen in [('features_v2', FEATURES_DIR), ('embeddings_cnn', EMB_DIR)]:
        destino = f'{local}/{nombre}'
        destinos[nombre] = destino
        n_local = len(glob.glob(f'{destino}/*/*.npz'))
        n_drive = len(glob.glob(f'{origen}/*/*.npz'))
        if n_local >= n_drive > 0:
            print(f'  {nombre}: ya esta en local ({n_local} archivos)')
            continue
        t0 = time.time()
        print(f'  copiando {nombre} ({n_drive} archivos) a {destino} ...')
        shutil.copytree(origen, destino, dirs_exist_ok=True)
        print(f'    listo en {(time.time()-t0)/60:.1f} min')
    return destinos['features_v2'], destinos['embeddings_cnn']

def load_aligned_split(split, features_dir, emb_dir):
    geom_files = {os.path.splitext(os.path.basename(f))[0]: f
                  for f in glob.glob(f'{features_dir}/{split}/*.npz')}
    emb_files  = {os.path.splitext(os.path.basename(f))[0]: f
                  for f in glob.glob(f'{emb_dir}/{split}/*.npz')}
    common_ids = sorted(set(geom_files) & set(emb_files))
    print(f'{split}: {len(geom_files)} geometricas, {len(emb_files)} embeddings, '
          f'{len(common_ids)} en comun')

    X_geom, X_emb, y3, eng, conf, frus, ids = [], [], [], [], [], [], []
    feat_names = None
    for cid in tqdm(common_ids, desc=f'  {split}', unit='clip'):
        dg = np.load(geom_files[cid], allow_pickle=True)
        de = np.load(emb_files[cid], allow_pickle=True)
        seq = dg['sequence'].astype(np.float32)
        emb = de['embeddings'].astype(np.float32)
        if seq.shape != (N_FRAMES, 22) or emb.shape != (N_EMB_FRAMES, EMB_DIM):
            continue
        if feat_names is None:
            feat_names = list(dg['feature_names'])
        X_geom.append(seq); X_emb.append(emb)
        b = int(dg['boredom'])
        y3.append(0 if b == 0 else (1 if b == 1 else 2))
        eng.append(int(dg['engagement'])); conf.append(int(dg['confusion']))
        frus.append(int(dg['frustration']))
        ids.append(cid)
    return (np.stack(X_geom), np.stack(X_emb), np.array(y3, dtype=np.int64),
            np.array(eng), np.array(conf), np.array(frus), np.array(ids), feat_names)

# ---------------------------------------------------------------- camino 1: cache
if USAR_CACHE and os.path.exists(CACHE_NPZ):
    t0 = time.time()
    d = np.load(CACHE_NPZ, allow_pickle=True)
    Xg_tr, Xe_tr, y3_tr, eng_tr, conf_tr, frus_tr, id_tr = (
        d['Xg_tr'], d['Xe_tr'], d['y3_tr'], d['eng_tr'], d['conf_tr'], d['frus_tr'], d['id_tr'])
    Xg_va, Xe_va, y3_va, eng_va, conf_va, frus_va, id_va = (
        d['Xg_va'], d['Xe_va'], d['y3_va'], d['eng_va'], d['conf_va'], d['frus_va'], d['id_va'])
    Xg_te, Xe_te, y3_te, eng_te, conf_te, frus_te, id_te = (
        d['Xg_te'], d['Xe_te'], d['y3_te'], d['eng_te'], d['conf_te'], d['frus_te'], d['id_te'])
    FEATS22 = list(d['feature_names'])
    print(f'Cargado de cache en {time.time()-t0:.1f}s: {CACHE_NPZ}')

# ------------------------------------------- camino 2: staging local + relectura
else:
    print('Sin cache -> staging local y lectura de los .npz individuales')
    FEAT_LOCAL, EMB_LOCAL = staging_local()
    Xg_tr, Xe_tr, y3_tr, eng_tr, conf_tr, frus_tr, id_tr, FEATS22 = load_aligned_split('Train', FEAT_LOCAL, EMB_LOCAL)
    Xg_va, Xe_va, y3_va, eng_va, conf_va, frus_va, id_va, _       = load_aligned_split('Validation', FEAT_LOCAL, EMB_LOCAL)
    Xg_te, Xe_te, y3_te, eng_te, conf_te, frus_te, id_te, _       = load_aligned_split('Test', FEAT_LOCAL, EMB_LOCAL)

    t0 = time.time()
    np.savez(CACHE_NPZ,
             Xg_tr=Xg_tr, Xe_tr=Xe_tr, y3_tr=y3_tr, eng_tr=eng_tr, conf_tr=conf_tr, frus_tr=frus_tr, id_tr=id_tr,
             Xg_va=Xg_va, Xe_va=Xe_va, y3_va=y3_va, eng_va=eng_va, conf_va=conf_va, frus_va=frus_va, id_va=id_va,
             Xg_te=Xg_te, Xe_te=Xe_te, y3_te=y3_te, eng_te=eng_te, conf_te=conf_te, frus_te=frus_te, id_te=id_te,
             feature_names=np.array(FEATS22, dtype=object))
    print(f'Cache guardada en {(time.time()-t0)/60:.1f} min: {CACHE_NPZ} '
          f'({os.path.getsize(CACHE_NPZ)/1e6:.0f} MB)')

print(f'\nTrain {Xg_tr.shape} + {Xe_tr.shape} | Validation {Xg_va.shape} | Test {Xg_te.shape}')

In [ ]:
# Re-split por participante (Train+Validation), Test oficial intacto -- igual criterio que Fase 3.0/3.1
Xg_pool = np.concatenate([Xg_tr, Xg_va]); Xe_pool = np.concatenate([Xe_tr, Xe_va])
y3_pool = np.concatenate([y3_tr, y3_va])
eng_pool = np.concatenate([eng_tr, eng_va]); conf_pool = np.concatenate([conf_tr, conf_va]); frus_pool = np.concatenate([frus_tr, frus_va])
id_pool = np.concatenate([id_tr, id_va])
participant_pool = np.array([cid[:6] for cid in id_pool])
y_bin_pool = (y3_pool > 0).astype(np.int64)

def interpolate_sequence(seq):
    df = pd.DataFrame(seq)
    df = df.interpolate(method='linear', axis=0, limit_direction='both')
    return df.ffill().bfill().values.astype(np.float32)

def clean_geom(X, train_medians=None, nan_frame_thresh=0.30):
    keep, X_clean = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

def clean_emb(X, train_medians=None, nan_frame_thresh=0.5):
    """Frames sin rostro detectado -> NaN (ver Fase 3.2a). Interpolar igual que geometricas;
    umbral mas laxo (50%) porque el muestreo es disperso (16 frames) y perder 1-2 pesa mas."""
    keep, X_clean = [], []
    for i in range(len(X)):
        if np.isnan(X[i]).any(axis=1).mean() > nan_frame_thresh:
            continue
        seq = interpolate_sequence(X[i])
        if np.isnan(seq).any() and train_medians is not None:
            idx = np.where(np.isnan(seq))
            seq[idx] = np.take(train_medians, idx[1])
        X_clean.append(seq); keep.append(i)
    return np.stack(X_clean), np.array(keep)

delta_idx = [FEATS22.index(f) for f in DELTA_FEATURES]
def add_deltas(X, idx):
    d_ = np.zeros_like(X[:, :, idx])
    d_[:, 1:, :] = np.diff(X[:, :, idx], axis=1)
    return np.concatenate([X, d_], axis=-1)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_idx0, va_idx0 = next(sgkf.split(Xg_pool, y_bin_pool, groups=participant_pool))
print(f'Pool: {len(id_pool)} clips | train_split={len(tr_idx0)} val_split={len(va_idx0)}')

In [ ]:
# Limpieza + deltas + escalado (geometricas) -- fit SOLO en el split de train
geom_medians = np.nanmedian(Xg_pool[tr_idx0].reshape(-1, 22), axis=0)
Xg_tr_c, keep_tr_g = clean_geom(Xg_pool[tr_idx0], geom_medians)
Xg_va_c, keep_va_g = clean_geom(Xg_pool[va_idx0], geom_medians)
Xg_tr_d = add_deltas(Xg_tr_c, delta_idx); Xg_va_d = add_deltas(Xg_va_c, delta_idx)
geom_scaler = StandardScaler().fit(Xg_tr_d.reshape(-1, 32))
def scale_geom(X):
    N, T, F = X.shape
    return geom_scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
Xg_tr_s, Xg_va_s = scale_geom(Xg_tr_d), scale_geom(Xg_va_d)

# Limpieza + escalado (embeddings) -- fit SOLO en el split de train
emb_medians = np.nanmedian(Xe_pool[tr_idx0].reshape(-1, 128), axis=0)
Xe_tr_c, keep_tr_e = clean_emb(Xe_pool[tr_idx0], emb_medians)
Xe_va_c, keep_va_e = clean_emb(Xe_pool[va_idx0], emb_medians)
emb_scaler = StandardScaler().fit(Xe_tr_c.reshape(-1, 128))
def scale_emb(X):
    N, T, F = X.shape
    return emb_scaler.transform(X.reshape(-1, F)).reshape(N, T, F).astype(np.float32)
Xe_tr_s, Xe_va_s = scale_emb(Xe_tr_c), scale_emb(Xe_va_c)

# Interseccion de indices validos entre ambas fuentes (clean_geom y clean_emb pueden descartar clips distintos)
keep_tr = np.intersect1d(keep_tr_g, keep_tr_e)
keep_va = np.intersect1d(keep_va_g, keep_va_e)
def reindex(X_full, keep_full, keep_final):
    pos = {k: i for i, k in enumerate(keep_full)}
    idx = [pos[k] for k in keep_final]
    return X_full[idx]
Xg_tr_s = reindex(Xg_tr_s, keep_tr_g, keep_tr); Xe_tr_s = reindex(Xe_tr_s, keep_tr_e, keep_tr)
Xg_va_s = reindex(Xg_va_s, keep_va_g, keep_va); Xe_va_s = reindex(Xe_va_s, keep_va_e, keep_va)
y_tr = y_bin_pool[tr_idx0][keep_tr]; y_va = y_bin_pool[va_idx0][keep_va]
print(f'Train final: {len(y_tr)} | Val final: {len(y_va)}')

# Test oficial: mismo tratamiento con estadisticos del train re-split
Xg_te_c, keep_te_g = clean_geom(Xg_te, geom_medians)
Xg_te_d = add_deltas(Xg_te_c, delta_idx); Xg_te_s = scale_geom(Xg_te_d)
Xe_te_c, keep_te_e = clean_emb(Xe_te, emb_medians)
Xe_te_s = scale_emb(Xe_te_c)
keep_te = np.intersect1d(keep_te_g, keep_te_e)
Xg_te_s = reindex(Xg_te_s, keep_te_g, keep_te); Xe_te_s = reindex(Xe_te_s, keep_te_e, keep_te)
y_bin_te_full = (y3_te > 0).astype(np.int64)
y_te = y_bin_te_full[keep_te]

trivial_test = max(Counter(y_te.tolist()).values()) / len(y_te)
print(f'Test oficial: {len(y_te)} clips | trivial={trivial_test:.4f}')

## 2. Arquitectura de fusión (§3.3 del plan)

Dos ramas: embeddings CNN (16×128, upsample a 60) + geométricas (60×32, `BatchNorm`) → concat por frame → TCN (igual que Fase 3.1) → attention pooling → cabezas binaria + ordinal (misma solución `Dense(K-1)` de Fase 3.1 por compatibilidad TFLite, ver ese notebook para el detalle del hallazgo).

**Adaptación a Keras 3.** Igual que en Fase 3.1: capas custom con sub-capas en `__init__` + `compute_output_shape`/`get_config`, y **nombres explícitos en todas las capas** para que la copia de pesos al modelo de export sea verificable (con nombres autogenerados la copia por nombre fallaba en silencio y el tronco TCN se exportaba con pesos aleatorios).

In [ ]:
@keras.saving.register_keras_serializable(package='sparkle')
class AttentionPooling(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = layers.Dense(1, name='attn_score')
    def call(self, x):
        scores = self.score_dense(x)
        weights = tf.nn.softmax(scores, axis=1)
        return tf.reduce_sum(x * weights, axis=1), tf.squeeze(weights, axis=-1)
    # Sin compute_output_shape: devuelve DOS tensores y Keras 3 interpretaria una tupla
    # como una sola forma. Omitiendolo, infiere las dos salidas trazando call().

@keras.saving.register_keras_serializable(package='sparkle')
class CoralHead(layers.Layer):
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        self.dense = layers.Dense(num_classes - 1, name='coral_thresholds')
    def call(self, x):
        return tf.nn.sigmoid(self.dense(x))
    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.num_classes - 1)
    def get_config(self):
        cfg = super().get_config(); cfg.update({'num_classes': self.num_classes}); return cfg

def tcn_residual_block(x, filters, kernel_size, dilation_rate, dropout=0.3, l2=1e-4, prefix='tcn'):
    reg = keras.regularizers.l2(l2) if l2 else None
    prev = x
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
                      kernel_regularizer=reg, name=f'{prefix}_conv1')(x)
    h = layers.BatchNormalization(name=f'{prefix}_bn1')(h)
    h = layers.Activation('relu', name=f'{prefix}_act1')(h)
    h = layers.Dropout(dropout, name=f'{prefix}_do1')(h)
    h = layers.Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
                      kernel_regularizer=reg, name=f'{prefix}_conv2')(h)
    h = layers.BatchNormalization(name=f'{prefix}_bn2')(h)
    h = layers.Activation('relu', name=f'{prefix}_act2')(h)
    h = layers.Dropout(dropout, name=f'{prefix}_do2')(h)
    if prev.shape[-1] != filters:
        prev = layers.Conv1D(filters, 1, padding='same', name=f'{prefix}_proj')(prev)
    return layers.Activation('relu', name=f'{prefix}_out')(
        layers.Add(name=f'{prefix}_add')([prev, h]))

def ordinal_loss(y_true_3level, p_hat, n_classes=3):
    y_true_3level = tf.cast(y_true_3level, tf.float32)
    losses = [keras.losses.binary_crossentropy(tf.cast(y_true_3level > k, tf.float32), p_hat[:, k])
              for k in range(n_classes - 1)]
    return tf.reduce_mean(tf.add_n(losses) / (n_classes - 1))

def build_fusion_model(input_mode, n_geom_feats=32, n_emb_frames=N_EMB_FRAMES, emb_dim=EMB_DIM,
                       n_frames=N_FRAMES, filters=64, dilations=(1, 2, 4, 8), dropout=0.3,
                       l2=1e-4, batch_size=None, export=False):
    '''input_mode: fusion | geom_only | emb_only -- para la ablacion de la seccion 3.
    export=True: sin dropout ni L2 y con batch fijo, para la conversion a TFLite.'''
    if export:
        dropout, l2 = 0.0, 0.0
    inputs, branches = [], []
    if input_mode in ('fusion', 'emb_only'):
        inp_emb = keras.Input(shape=(n_emb_frames, emb_dim), batch_size=batch_size,
                              name='embeddings_cnn')
        inputs.append(inp_emb)
        emb_up = layers.UpSampling1D(size=n_frames // n_emb_frames, name='emb_upsample')(inp_emb)
        cur_len = emb_up.shape[1]
        if cur_len < n_frames:
            emb_up = layers.ZeroPadding1D((0, n_frames - cur_len), name='emb_pad')(emb_up)
        elif cur_len > n_frames:
            emb_up = layers.Cropping1D((0, cur_len - n_frames), name='emb_crop')(emb_up)
        branches.append(emb_up)
    if input_mode in ('fusion', 'geom_only'):
        inp_geom = keras.Input(shape=(n_frames, n_geom_feats), batch_size=batch_size,
                               name='geometricas')
        inputs.append(inp_geom)
        branches.append(layers.BatchNormalization(name='geom_bn')(inp_geom))

    x = branches[0] if len(branches) == 1 else layers.Concatenate(axis=-1, name='concat')(branches)
    for i, d in enumerate(dilations):
        x = tcn_residual_block(x, filters, 3, d, dropout=dropout, l2=l2, prefix=f'tcn{i}')
    pooled, attn_w = AttentionPooling(name='attention_pooling')(x)
    h = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(l2) if l2 else None,
                     name='head_dense')(pooled)
    if not export:
        h = layers.Dropout(dropout, name='head_dropout')(h)
    binary_out = layers.Dense(1, activation='sigmoid', name='binary')(h)
    coral_out = CoralHead(3, name='coral')(h)
    outputs = [binary_out, coral_out, attn_w] if export else [binary_out, coral_out]
    return keras.Model(inputs, outputs, name=f'{input_mode}{"_export" if export else ""}')

for m in MODES:
    _m = build_fusion_model(m)
    print(f'{m:>10}: {_m.count_params():,} parametros')
    del _m

## 3. Entrenamiento y ablación — embeddings-solo vs. geométricas-solo vs. fusión

Esta tabla es la evidencia central del plan: si `fusion` supera claramente a `geom_only` (que ya se midió en Fase 3.0/3.1 en el orden de 55-63%), confirma que la apariencia es la palanca que faltaba.

Igual que en Fase 3.1, el `train_step` con `@tf.function` se define dentro de la función de entrenamiento (una por combinación modo × semilla) para que las variables de estado del optimizador se creen siempre en el primer trace. Los resultados se escriben al CSV de Drive **después de cada semilla**, así una desconexión de Colab no borra las 14 corridas anteriores.

> ⚠️ **Antes de correr esta celda: completá `CFG_FUSION` con la configuración ganadora del diagnóstico de Fase 3.1.**
>
> Este notebook usa el mismo tronco TCN y el mismo loop de entrenamiento manual que 3.1. La primera corrida de 3.1 (lr 1e-3, sin scheduler, sin piso de épocas) quedó por debajo del BiGRU y en 2 de 5 semillas el mejor `val_f1` cayó en la época 0 — `restore_best_weights` devolvía pesos sin entrenar. Si arrastrás ese protocolo acá, **no vas a poder distinguir "la apariencia no aporta" de "el tronco está mal entrenado"**, y esa distinción es exactamente lo que este experimento tiene que resolver.
>
> Control de sanidad al terminar: mirá la columna `mejor_epoca` en los resultados. Si queda pegada a `min_epochs` (10), el modelo no está aprendiendo, solo se lo está frenando antes.

In [ ]:
def smoothed_class_weights(y, beta=0.5):
    counts = Counter(y.tolist()); total = len(y)
    w = {int(c): (total / counts[c]) ** beta for c in counts}
    mean_w = np.mean(list(w.values()))
    return {c: w[c] / mean_w for c in w}

def get_inputs(mode, Xe, Xg):
    return {'fusion': [Xe, Xg], 'emb_only': [Xe], 'geom_only': [Xg]}[mode]

def as_model_input(x_list):
    return x_list if len(x_list) > 1 else x_list[0]

# ---------------------------------------------------------------------------
# PROTOCOLO DE ENTRENAMIENTO -- copiar aca la configuracion ganadora del
# diagnostico de Fase 3.1 (celda "3.1 Diagnostico" de ese notebook).
#
# La primera corrida de 3.1 con lr=1e-3, aux=0.1, sin scheduler y sin piso de
# epocas dio 0.5824 +/- 0.0277 y en 2 de 5 semillas el mejor val_f1 cayo en la
# EPOCA 0: restore_best_weights devolvia pesos sin entrenar. Este notebook usa
# el MISMO tronco TCN y el MISMO loop manual, asi que hereda el problema si no
# se corrige. Sin esto no se puede distinguir "la apariencia no aporta" de
# "el tronco esta mal entrenado" -- que es justo lo que decide el proyecto.
# ---------------------------------------------------------------------------
CFG_BASE = dict(
    lr=5e-4,             # era 1e-3
    ord_weight=0.5,
    min_epochs=10,       # no habilitar early stopping antes de esta epoca
    lr_patience=6,       # ReduceLROnPlateau manual
    lr_factor=0.5,
    lr_min=1e-5,
    label_smoothing=0.1,
    dropout=0.3,
    filters=64,
)
CFG_FUSION = dict()      # <-- pegar aca lo que gano en 3.1, p.ej. dict(filters=32, dropout=0.35)

def train_and_eval_fusion(mode, seed, cfg=None, epochs=MAX_EPOCHS, patience=PATIENCE,
                          batch_size=BATCH_SIZE, verbose=True):
    cfg = {**CFG_BASE, **CFG_FUSION, **(cfg or {})}
    try:
        keras.utils.set_random_seed(seed)
    except AttributeError:
        pass
    tf.random.set_seed(seed); np.random.seed(seed)

    model = build_fusion_model(mode, filters=cfg['filters'], dropout=cfg['dropout'])
    optimizer = keras.optimizers.Adam(learning_rate=cfg['lr'], clipnorm=1.0)
    cw = smoothed_class_weights(y_tr, beta=0.5)
    sw_tr = np.array([cw[int(v)] for v in y_tr], dtype='float32')

    Xtr_in = get_inputs(mode, Xe_tr_s, Xg_tr_s)
    Xva_in = as_model_input(get_inputs(mode, Xe_va_s, Xg_va_s))
    y3_tr_pool = y3_pool[tr_idx0][keep_tr]      # etiqueta de 3 niveles para la perdida ordinal

    ds = tf.data.Dataset.from_tensor_slices((tuple(Xtr_in), y_tr.astype('float32'),
                                             y3_tr_pool.astype('int64'), sw_tr))
    ds = ds.shuffle(4096, seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    ord_w, ls = cfg['ord_weight'], cfg['label_smoothing']

    @tf.function
    def train_step(Xb, yb, y3b, swb):
        Xb_list = list(Xb) if isinstance(Xb, (tuple, list)) else [Xb]
        with tf.GradientTape() as tape:
            p_bin, p_coral = model(Xb_list if len(Xb_list) > 1 else Xb_list[0], training=True)
            l_bin = tf.reduce_mean(swb * keras.losses.binary_crossentropy(
                yb[:, None], p_bin, label_smoothing=ls))
            l_ord = ordinal_loss(y3b, p_coral)
            total = l_bin + ord_w * l_ord
            if model.losses:
                total += tf.add_n(model.losses)
        grads = tape.gradient(total, model.trainable_variables)
        grads, _ = tf.clip_by_global_norm(grads, 1.0)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return total

    def set_lr(v):
        try:
            optimizer.learning_rate.assign(v)
        except AttributeError:
            optimizer.learning_rate = v

    best_f1, wait, lr_wait, best_w, best_epoch = -1.0, 0, 0, None, -1
    t0 = time.time()
    for epoch in range(epochs):
        for batch in ds:
            train_step(*batch)
        p_val = (model.predict(Xva_in, verbose=0, batch_size=256)[0].ravel() > 0.5).astype(int)
        f1_val = f1_score(y_va, p_val, average='macro')

        # Igual que en 3.1: antes de min_epochs no se registra "mejor" ni corre la
        # paciencia, para que restore_best_weights no devuelva un modelo sin entrenar.
        if epoch < cfg['min_epochs']:
            continue

        if best_w is None or f1_val > best_f1 + 1e-4:
            best_f1, wait, lr_wait = f1_val, 0, 0
            best_w, best_epoch = model.get_weights(), epoch
        else:
            wait += 1; lr_wait += 1
            if lr_wait >= cfg['lr_patience']:
                nueva = max(float(optimizer.learning_rate) * cfg['lr_factor'], cfg['lr_min'])
                set_lr(nueva); lr_wait = 0
        if verbose and (epoch % 20 == 0 or wait >= patience):
            print(f'    epoca {epoch:3d} | val_f1={f1_val:.4f} | mejor={best_f1:.4f} '
                  f'(ep {best_epoch}) | {(time.time()-t0)/60:.1f} min')
        if wait >= patience:
            break
    model.set_weights(best_w)
    return model, best_f1, epoch + 1, (time.time() - t0) / 60, best_epoch

csv_abl = f'{CKPT_DIR}/fase3_2_ablacion_5semillas.csv'
if os.path.exists(csv_abl):
    os.remove(csv_abl)

ablation_results = []
models_by_mode = {}
for mode in MODES:
    mode_results = []
    for seed in SEEDS:
        print(f'--- {mode} | semilla {seed} ---')
        model, best_val_f1, stopped_epoch, minutos, best_epoch = train_and_eval_fusion(mode, seed)
        Xte_in = as_model_input(get_inputs(mode, Xe_te_s, Xg_te_s))
        p_te = (model.predict(Xte_in, verbose=0, batch_size=256)[0].ravel() > 0.5).astype(int)
        acc = accuracy_score(y_te, p_te)
        r = {'mode': mode, 'seed': seed, 'acc': acc,
             'f1_macro': f1_score(y_te, p_te, average='macro'),
             'balanced_acc': balanced_accuracy_score(y_te, p_te),
             'gap_vs_trivial': acc - trivial_test, 'stopped_epoch': stopped_epoch,
             'mejor_epoca': best_epoch,
             'best_val_f1': best_val_f1, 'minutos': round(minutos, 2)}
        mode_results.append(r); ablation_results.append(r)
        pd.DataFrame([r]).to_csv(csv_abl, mode='a', header=not os.path.exists(csv_abl), index=False)
        print(f"[{mode}] seed={seed}: acc={acc:.4f} gap={r['gap_vs_trivial']:+.4f} ({minutos:.1f} min)")
        if seed == SEEDS[0]:
            models_by_mode[mode] = model
            model.save(f'{CKPT_DIR}/{mode}_seed0.keras')
    df_mode = pd.DataFrame(mode_results)
    print(f"=== {mode}: acc={df_mode['acc'].mean():.4f}+/-{df_mode['acc'].std():.4f} "
          f"gap={df_mode['gap_vs_trivial'].mean():+.4f}+/-{df_mode['gap_vs_trivial'].std():.4f} ===\n")

df_ablation = pd.DataFrame(ablation_results)
resumen = df_ablation.groupby('mode')[['acc', 'f1_macro', 'balanced_acc', 'gap_vs_trivial']].agg(['mean', 'std'])
print(resumen)
resumen.to_csv(f'{CKPT_DIR}/fase3_2_ablacion_resumen.csv')
print(f'\nTrivial en Test: {trivial_test:.4f}')

### Lectura de la ablación — el experimento que decide el proyecto

La comparación relevante es `fusion` vs `geom_only` sobre el mismo Test oficial y el mismo re-split. La celda siguiente reporta la diferencia de medias con un intervalo bootstrap sobre las 5 semillas, para no leer una diferencia que esté dentro del ruido entre semillas.

In [ ]:
acc_fusion = df_ablation.query("mode == 'fusion'")['acc'].values
acc_geom   = df_ablation.query("mode == 'geom_only'")['acc'].values
acc_emb    = df_ablation.query("mode == 'emb_only'")['acc'].values

rng = np.random.default_rng(SEED)
boot = np.array([rng.choice(acc_fusion, len(acc_fusion)).mean() -
                 rng.choice(acc_geom, len(acc_geom)).mean() for _ in range(10000)])
lo, hi = np.percentile(boot, [2.5, 97.5])

print(f'geom_only : {acc_geom.mean():.4f} +/- {acc_geom.std():.4f}')
print(f'emb_only  : {acc_emb.mean():.4f} +/- {acc_emb.std():.4f}')
print(f'fusion    : {acc_fusion.mean():.4f} +/- {acc_fusion.std():.4f}  (trivial={trivial_test:.4f})')
print(f'\nfusion - geom_only = {acc_fusion.mean()-acc_geom.mean():+.4f} '
      f'[IC 95% bootstrap: {lo:+.4f}, {hi:+.4f}]')
print(f'Brecha de fusion vs trivial: {acc_fusion.mean()-trivial_test:+.4f} '
      f'(criterio del plan: > +0.12)')
print('\nCriterio de salida Fase 3.2:',
      'CUMPLE' if (acc_fusion.mean() >= 0.68 and acc_fusion.mean() - trivial_test > 0.12
                   and lo > 0) else 'NO CUMPLE con claridad')

best_model_fusion = models_by_mode['fusion']
Xte_in = as_model_input(get_inputs('fusion', Xe_te_s, Xg_te_s))
p_te_fusion = (best_model_fusion.predict(Xte_in, verbose=0, batch_size=256)[0].ravel() > 0.5).astype(int)
print('\n' + classification_report(y_te, p_te_fusion, target_names=['Atento','Desatencion'], digits=3))

## 4. Export a TFLite del modelo de fusión (2 modelos: extractor CNN + fusión temporal)

El extractor CNN (`Sparkle_V2_Fase3_2a`) ya se exporta por separado. Acá se exporta el modelo de fusión temporal (batch fijo=1, misma mitigación que en Fase 3.0/3.1 para evitar operadores dinámicos no soportados).

**Adaptación a Keras 3:** misma solución que en Fase 3.1 — `ExportArchive` con firma de batch fijo → SavedModel → `TFLiteConverter.from_saved_model`. Acá la firma tiene **dos** entradas (embeddings y geométricas), en el mismo orden en que se declaran los `keras.Input` del modelo de export.

In [ ]:
def keras_to_tflite(model, input_specs, representative_dataset=None, verbose=True):
    '''Keras 3 -> SavedModel (batch fijo, via ExportArchive) -> TFLite.'''
    sm_dir = tempfile.mkdtemp(prefix='sparkle_sm_')
    shutil.rmtree(sm_dir, ignore_errors=True)
    exported = False
    try:
        from keras.export import ExportArchive
        archive = ExportArchive(); archive.track(model)
        if len(input_specs) == 1:
            fn = lambda x: model(x, training=False)
        else:
            fn = lambda *xs: model(list(xs), training=False)
        archive.add_endpoint(name='serve', fn=fn, input_signature=list(input_specs))
        archive.write_out(sm_dir); exported = True
        if verbose:
            print('  SavedModel exportado con ExportArchive (batch fijo)')
    except Exception as e:
        print('  ExportArchive fallo:', repr(e)[:200])
        try:
            model.export(sm_dir); exported = True
        except Exception as e2:
            print('  model.export() fallo:', repr(e2)[:200])
    converter = (tf.lite.TFLiteConverter.from_saved_model(sm_dir) if exported
                 else tf.lite.TFLiteConverter.from_keras_model(model))
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    if representative_dataset is not None:
        converter.representative_dataset = representative_dataset
    out = converter.convert()
    shutil.rmtree(sm_dir, ignore_errors=True)
    return out

def copy_weights_by_name(src, dst, allow_missing=()):
    src_map = {l.name: l for l in src.layers}
    copied, missing = [], []
    for layer in dst.layers:
        if not layer.weights:
            continue
        s = src_map.get(layer.name)
        if s is None or not s.weights:
            missing.append(layer.name); continue
        try:
            layer.set_weights(s.get_weights()); copied.append(layer.name)
        except ValueError:
            missing.append(f'{layer.name}(shape)')
    print(f'  Pesos copiados: {len(copied)} capas | sin copiar: {missing or "ninguna"}')
    pendientes = [m for m in missing if m not in allow_missing]
    assert not pendientes, f'Capas sin pesos copiados: {pendientes}'
    return copied

# El modelo de export tiene que replicar la ARQUITECTURA del entrenado: si CFG_FUSION
# cambia `filters`, hay que propagarlo o no coincide ninguna shape del tronco TCN.
cfg_exp = {**CFG_BASE, **CFG_FUSION}
export_model = build_fusion_model('fusion', export=True, batch_size=1,
                                  filters=cfg_exp['filters'])
print(f'Modelo de export con filters={cfg_exp["filters"]} '
      f'({export_model.count_params():,} parametros)')
copy_weights_by_name(best_model_fusion, export_model)

specs = [tf.TensorSpec((1, N_EMB_FRAMES, EMB_DIM), tf.float32, name='embeddings_cnn'),
         tf.TensorSpec((1, N_FRAMES, 32), tf.float32, name='geometricas')]

tflite_float = keras_to_tflite(export_model, specs)
print(f'TFLite fusion float: {len(tflite_float)/1024:.1f} KB')

rng = np.random.default_rng(SEED)
idx_rep = rng.choice(len(Xe_tr_s), size=min(200, len(Xe_tr_s)), replace=False)
rep_emb, rep_geom = Xe_tr_s[idx_rep], Xg_tr_s[idx_rep]   # MISMOS indices: pares reales, no cruzados
def representative_dataset():
    for i in range(len(idx_rep)):
        yield [rep_emb[i:i+1].astype('float32'), rep_geom[i:i+1].astype('float32')]

tflite_int8 = keras_to_tflite(export_model, specs, representative_dataset)
print(f'TFLite fusion int8: {len(tflite_int8)/1024:.1f} KB')

with open(f'{CKPT_DIR}/fusion_model.tflite', 'wb') as f:
    f.write(tflite_int8)
with open(f'{CKPT_DIR}/fusion_model_float.tflite', 'wb') as f:
    f.write(tflite_float)

### Verificación del modelo TFLite de fusión

Se compara la salida binaria del intérprete TFLite contra la de Keras sobre las mismas muestras de Test. Las entradas se identifican por su forma (16×128 = embeddings, 60×32 = geométricas) en vez de por índice: con `from_saved_model`, el orden de entradas y salidas lo fija la firma exportada y no siempre coincide con el orden de declaración.

In [ ]:
interpreter = tf.lite.Interpreter(model_content=tflite_int8)
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()
out_det = interpreter.get_output_details()
print('Entradas:', [(d['name'], d['shape'].tolist()) for d in in_det])
print('Salidas :', [(d['name'], d['shape'].tolist()) for d in out_det])

emb_det  = next(d for d in in_det if tuple(d['shape'][1:]) == (N_EMB_FRAMES, EMB_DIM))
geom_det = next(d for d in in_det if tuple(d['shape'][1:]) == (N_FRAMES, 32))
bin_det  = next(d for d in out_det if tuple(d['shape'][1:]) == (1,))

def set_q(det, arr):
    if det['dtype'] == np.int8:
        s, zp = det['quantization']
        return np.clip(np.round(arr / s + zp), -128, 127).astype(np.int8)
    return arr.astype(np.float32)

n_check = min(300, len(y_te))
preds = np.zeros(n_check, dtype=np.float32)
for i in range(n_check):
    interpreter.set_tensor(emb_det['index'],  set_q(emb_det,  Xe_te_s[i:i+1]))
    interpreter.set_tensor(geom_det['index'], set_q(geom_det, Xg_te_s[i:i+1]))
    interpreter.invoke()
    o = interpreter.get_tensor(bin_det['index']).ravel()[0]
    if bin_det['dtype'] == np.int8:
        s_o, zp_o = bin_det['quantization']
        o = (float(o) - zp_o) * s_o
    preds[i] = o

acc_tflite = accuracy_score(y_te[:n_check], (preds > 0.5).astype(int))
acc_keras  = accuracy_score(y_te[:n_check], p_te_fusion[:n_check])
print(f'\nAccuracy TFLite int8 ({n_check} muestras): {acc_tflite:.4f}')
print(f'Accuracy Keras      ({n_check} muestras): {acc_keras:.4f}')
print(f'Delta por cuantizacion: {abs(acc_tflite-acc_keras):.4f} '
      f'({"OK, < 1,5 pts" if abs(acc_tflite-acc_keras) < 0.015 else "REVISAR: considerar QAT (Fase 3.3)"})')

extractor_tflite = f'{CKPT_DIR}/cnn_extractor.tflite'
tam_extractor = os.path.getsize(extractor_tflite)/1024 if os.path.exists(extractor_tflite) else None
with open(f'{CKPT_DIR}/fase3_2_tflite_reporte.json', 'w') as f:
    json.dump({'fusion_float_kb': round(len(tflite_float)/1024, 1),
               'fusion_int8_kb': round(len(tflite_int8)/1024, 1),
               'extractor_cnn_kb': tam_extractor,
               'acc_keras': float(acc_keras), 'acc_tflite_int8': float(acc_tflite),
               'delta': float(abs(acc_tflite - acc_keras)), 'n_muestras': int(n_check)}, f, indent=2)
print('\nReporte guardado en', f'{CKPT_DIR}/fase3_2_tflite_reporte.json')

## 5. Criterios de salida de Fase 3.2

| Métrica | Umbral del plan | Resultado |
| :--- | :--- | :--- |
| Accuracy binaria Test oficial, modo `fusion` (media 5 semillas) | ≥ 68-72% | ver tabla de la sección 3 |
| Brecha vs. trivial, modo `fusion` | > +12 pts | ver tabla de la sección 3 |
| `fusion` > `geom_only` de forma clara | — | evidencia de que la apariencia es la palanca principal (diagnóstico §0 del plan) |
| TFLite (extractor + fusión) | tamaño total < ~3-4 MB (backbone ~2MB + fusión <1MB) | ver sección 4 y el notebook 3.2a |

Si se cumple: pasar a **Fase 3.3** (cuantización int8 con QAT si hace falta, benchmark de latencia/energía en hardware real, evaluación final con 5 semillas + bootstrap CI + análisis por participante — ver Roadmap del plan). Este es, según el plan, el resultado que sostiene la tesis: *"mismo orden de exactitud que enfoques full-frame, sin que un solo píxel salga del dispositivo"* (§7).